# Bayesian optimisation of Extra Trees
Extra Trees led the comparable Phase 1 runs. This search uses the shared features, proxy labels and five fixed trip folds. It evaluates the chosen configuration on student-test only after locking selection. The evaluator partition is untouched.

In [1]:
from pathlib import Path
import hashlib, itertools, json, os, platform, time, warnings

# Run from the repository root, even when Jupyter opens in notebooks/.
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
os.chdir(root)
data_root = Path("processed")
assert (data_root / "schema.json").is_file(), "Prepare processed/ using 01_unpack_data.ipynb."
get_ipython().run_line_magic("run", "notebooks/00_shared_training.ipynb")


## Search space and budget
The published baseline and three diverse configurations initialise the Gaussian process. Expected improvement proposed two completed configurations. The initial plan was four proposals, and the trial budget was reduced at the user’s request. Work on trial 7 was interrupted and its partial fold results were excluded. The grid spans tree counts for CPU cost, depth and leaf size for regularisation, and features per split. The baseline’s sqrt setting selects three of the ten features.

In [2]:
import joblib
import numpy as np
import sklearn
import scipy
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel

RUN_ID = "bayesian_extra_trees_01"
workers = min(6, os.cpu_count() or 1)
output = Path("results") / RUN_ID
output.mkdir(parents=True, exist_ok=True)

parameter_names = ("n_estimators", "max_depth", "min_samples_leaf", "max_features")
grid = {
    "n_estimators": [64, 96, 128, 150, 192],
    "max_depth": [16, 20, 24, 28, 32],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": [3, 5, 7],
}
# sqrt(10) selects 3 features per split, so use 3 in GP coordinates.
baseline = (150, 24, 2, 3)
initial = [baseline, (64, 20, 4, 3), (96, 28, 2, 5), (96, 16, 8, 7)]
bayesian_trials = 2
pool = list(itertools.product(*(grid[key] for key in parameter_names)))
lower, upper = np.min(pool, axis=0), np.max(pool, axis=0)

def configuration(point):
    params = dict(zip(parameter_names, point))
    if point == baseline:
        params["max_features"] = "sqrt"
    return params

def write_json(path, data):
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(data, indent=2) + "\n")
    temp.replace(path)

manifest = {
    "objective": "mean all-observation trip-and-arc Macro F1 across five fixed folds",
    "train_trip_count": 80,
    "folds": list(range(5)),
    "search_space": grid,
    "initial_points": [list(p) for p in initial],
    "bayesian_trials": bayesian_trials,
    "surrogate": "Gaussian process, Matern 2.5 and white noise",
    "acquisition": "expected improvement, xi=0.001",
    "seed": SEED,
    "workers": workers,
    "python": platform.python_version(),
    "scikit_learn": sklearn.__version__,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "notebook_sha256": {
        name: hashlib.sha256("".join("".join(cell["source"]) for cell in
            json.loads((Path("notebooks") / name).read_text())["cells"]
            if cell["cell_type"] == "code").encode()).hexdigest()
        for name in ["04_bayesian_extra_trees.ipynb", "00_shared_training.ipynb"]
    },
    "data_sha256": {str(p.relative_to(data_root)): hashlib.sha256(p.read_bytes()).hexdigest()
                    for p in sorted(data_root.glob("*/*.npz"))},
}
manifest_file = output / "search_configuration.json"
if manifest_file.exists() and json.loads(manifest_file.read_text()) != manifest:
    raise ValueError("This run ID belongs to a different search or dataset. Change RUN_ID.")
write_json(manifest_file, manifest)


## Five-fold search on training trips
Optimise mean observation-level arc Macro F1 across the fixed folds. Trial metrics include candidate misses. Completed trials are checkpointed. Reuse the run ID only with identical code and data.

In [3]:
def select_next_trial(history):
    observed = [tuple(trial["point"]) for trial in history]
    scores = np.array([trial["mean_macro_f1"] for trial in history])
    options = [point for point in pool if point not in observed]
    gp = GaussianProcessRegressor(
        kernel=Matern(length_scale=0.35, nu=2.5) + WhiteKernel(noise_level=1e-5),
        optimizer=None, normalize_y=True, random_state=SEED,
    )
    gp.fit((np.array(observed) - lower) / (upper - lower), scores)
    mean, std = gp.predict((np.array(options) - lower) / (upper - lower), return_std=True)
    gain = mean - max(scores) - 0.001
    z = gain / np.maximum(std, 1e-12)
    expected_improvement = gain * norm.cdf(z) + std * norm.pdf(z)
    idx = int(np.argmax(expected_improvement))
    return options[idx], float(expected_improvement[idx])

def fit_and_score(point, fitting, evaluation):
    model = make_model("extra_trees").set_params(n_jobs=workers, **configuration(point))
    train_x = fitting["X"].reshape(-1, fitting["X"].shape[-1])
    test_x = evaluation["X"].reshape(-1, evaluation["X"].shape[-1])
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        start = time.perf_counter()
        model.fit(train_x, fitting["y"].ravel())
        fit_seconds = time.perf_counter() - start
        start = time.perf_counter()
        score = model.predict_proba(test_x)[:, list(model.classes_).index(1)]
        predict_seconds = time.perf_counter() - start
    metrics, arcs = score_predictions(evaluation, score)
    record = {
        "all_points": metrics["all_points"],
        "reliable_points": metrics["reliable_points"],
        "reliable_coverage": metrics["reliable_coverage"],
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
        "inference_ms_per_observation": 1000 * predict_seconds / len(arcs),
        "warnings": [str(w.message) for w in captured],
    }
    return model, score, arcs, record

history_path = output / "trials.json"
history = json.loads(history_path.read_text()) if history_path.exists() else []
for checkpointed in history:
    print(f"Saved trial {checkpointed['number']}: {checkpointed['phase']}, CV Macro F1={checkpointed['mean_macro_f1']:.5f}")
folds = [load_training(data_root, fold) for fold in range(5)]
for fold, (fitting, validation) in enumerate(folds):
    assert not set(fitting["trip_id"]) & set(validation["trip_id"])
    assert len(set(validation["trip_id"])) == 16

for trial_number in range(len(history), len(initial) + bayesian_trials):
    if trial_number < len(initial):
        point, acquisition = initial[trial_number], None
        phase = "published_baseline" if trial_number == 0 else "initial_design"
    else:
        point, acquisition = select_next_trial(history)
        phase = "bayesian_expected_improvement"
    print(f"Trial {trial_number + 1} {phase} {configuration(point)}", flush=True)
    start = time.perf_counter()
    records = []
    for fold, (fitting, validation) in enumerate(folds):
        fitted, _, _, record = fit_and_score(point, fitting, validation)
        del fitted
        score = record["all_points"]["macro_f1"]
        records.append({"fold": fold, **record})
        print(f"  fold {fold}: F1={score:.5f}, fit={record['fit_seconds']:.1f}s", flush=True)
    scores = [r["all_points"]["macro_f1"] for r in records]
    trial = {"number": trial_number + 1, "phase": phase, "point": list(point),
             "parameters": configuration(point), "expected_improvement": acquisition,
             "mean_macro_f1": float(np.mean(scores)), "std_macro_f1": float(np.std(scores)),
             "wall_seconds": time.perf_counter() - start, "folds": records}
    history.append(trial)
    write_json(history_path, history)
    print(f"  mean grouped CV Macro F1={trial['mean_macro_f1']:.5f}", flush=True)
del folds, fitting, validation


Saved trial 1: published_baseline, CV Macro F1=0.65852
Saved trial 2: initial_design, CV Macro F1=0.65645
Saved trial 3: initial_design, CV Macro F1=0.65697
Saved trial 4: initial_design, CV Macro F1=0.65910
Saved trial 5: bayesian_expected_improvement, CV Macro F1=0.65915
Saved trial 6: bayesian_expected_improvement, CV Macro F1=0.65912


## Locked selection and student-test comparison
Refit the published baseline and the CV-selected configuration. Save both models and per-observation candidate scores in trip order for HMM evaluation. Student-test numbers measure agreement with proxy labels. Timing covers classifier inference on prepared inputs.

In [4]:
# Lock the winner on CV before evaluating the student-test labels.
best_trial = max(history, key=lambda trial: trial["mean_macro_f1"])
write_json(output / "selection.json", {
    "selected_trial": best_trial["number"],
    "selected_parameters": best_trial["parameters"],
    "cv_macro_f1": best_trial["mean_macro_f1"],
    "search_wall_seconds": sum(trial["wall_seconds"] for trial in history),
})
fitting, evaluation = load_training(data_root)
comparison = {}
for label, trial in [("untuned", history[0]), ("tuned", best_trial)]:
    point = tuple(trial["point"])
    fitted, probabilities, arcs, record = fit_and_score(point, fitting, evaluation)
    joblib.dump(fitted, output / f"{label}_model.joblib", compress=3)
    np.savez_compressed(
        output / f"{label}_student_test_predictions.npz",
        positive_probability=probabilities.reshape(evaluation["y"].shape),
        predicted_arc_id=arcs,
        candidate_arc_ids=evaluation["candidate_arc_ids"],
        reference_arc_id=evaluation["reference_arc_id"],
        trip_id=evaluation["trip_id"],
        label_reliable=evaluation["label_reliable"],
    )
    comparison[label] = {"trial": trial["number"], "parameters": trial["parameters"],
                         "cv_macro_f1_mean": trial["mean_macro_f1"],
                         "cv_macro_f1_sd": trial["std_macro_f1"], **record}
    del fitted
write_json(output / "comparison.json", comparison)
print("Selected configuration:", best_trial["parameters"])
for label, row in comparison.items():
    m = row["all_points"]
    print(f"{label}: CV F1={row['cv_macro_f1_mean']:.5f}, "
          f"student Top-1={m['top1_accuracy']:.4%}, student F1={m['macro_f1']:.5f}, "
          f"log loss={m['categorical_log_loss_with_miss_state']:.5f}, "
          f"predict={row['inference_ms_per_observation']:.4f} ms/observation")


Selected configuration: {'n_estimators': 150, 'max_depth': 16, 'min_samples_leaf': 8, 'max_features': 7}
untuned: CV F1=0.65852, student Top-1=81.1065%, student F1=0.63245, log loss=0.62583, predict=0.0231 ms/observation
tuned: CV F1=0.65915, student Top-1=81.1795%, student F1=0.63579, log loss=0.62239, predict=0.0187 ms/observation


## Measured result

Trial 5 won on grouped CV with 150 trees, maximum depth 16, minimum leaf size 8 and seven features considered per split. Its mean CV Macro F1 was 0.65915 versus 0.65852 for the baseline. The improvement was small relative to scores varying across folds.

On the same 19,160 student-test observations, Top-1 rose from 81.1065% to 81.1795% (14 additional matches), arc Macro F1 rose from 0.63245 to 0.63579 and categorical log loss fell from 0.62583 to 0.62239. Those are classifier-only scores against the shared proxy labels.

Use the tuned score array and candidate arc IDs for the first HMM comparison. Score the same tuned classifier with and without Viterbi on identical observations and report route continuity and sequence metrics separately.
